In [1]:
from pyspark.sql import SparkSession
import getpass

In [2]:
username = getpass.getuser()

In [3]:
spark = SparkSession.builder. \
config('spark.ui.port','0'). \
config("spark.sql.warehouse.dir", f"/user/{username}/warehouse"). \
enableHiveSupport(). \
master('yarn'). \
getOrCreate()

In [4]:
order_schema = "order_id int, order_date timestamp, customer_id long, order_status string"
read_df = spark.read.format('csv').schema(order_schema).load('datasets/retail_db_orders_part-00000')

In [5]:
read_df.show(10)

+--------+-------------------+-----------+---------------+
|order_id|         order_date|customer_id|   order_status|
+--------+-------------------+-----------+---------------+
|       1|2013-07-25 00:00:00|      11599|         CLOSED|
|       2|2013-07-25 00:00:00|        256|PENDING_PAYMENT|
|       3|2013-07-25 00:00:00|      12111|       COMPLETE|
|       4|2013-07-25 00:00:00|       8827|         CLOSED|
|       5|2013-07-25 00:00:00|      11318|       COMPLETE|
|       6|2013-07-25 00:00:00|       7130|       COMPLETE|
|       7|2013-07-25 00:00:00|       4530|       COMPLETE|
|       8|2013-07-25 00:00:00|       2911|     PROCESSING|
|       9|2013-07-25 00:00:00|       5657|PENDING_PAYMENT|
|      10|2013-07-25 00:00:00|       5648|PENDING_PAYMENT|
+--------+-------------------+-----------+---------------+
only showing top 10 rows



In [6]:
read_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- order_status: string (nullable = true)



In [7]:
read_df.createOrReplaceTempView('order_view')

In [8]:
spark.sql("select * from order_view limit 5")

order_id,order_date,customer_id,order_status
1,2013-07-25 00:00:00,11599,CLOSED
2,2013-07-25 00:00:00,256,PENDING_PAYMENT
3,2013-07-25 00:00:00,12111,COMPLETE
4,2013-07-25 00:00:00,8827,CLOSED
5,2013-07-25 00:00:00,11318,COMPLETE


In [9]:
spark.sql("show databases").filter("namespace = 'big_data_025320'")

namespace
big_data_025320


In [10]:
spark.sql("show tables").filter("database != 'default'")

database,tableName,isTemporary
,order_view,true


In [11]:
spark.sql("""
    CREATE TABLE external_table_order_relative
    USING CSV
    LOCATION 'data'
    AS SELECT * from order_view
""")

Py4JJavaError: An error occurred while calling o45.sql.
: org.apache.hadoop.security.AccessControlException: Permission denied: user=itv025320, access=WRITE, inode="/user/hive/warehouse":hive:students:drwxr-xr-x
	at org.apache.hadoop.hdfs.server.namenode.FSPermissionChecker.check(FSPermissionChecker.java:496)
	at org.apache.hadoop.hdfs.server.namenode.FSPermissionChecker.checkPermission(FSPermissionChecker.java:339)
	at org.apache.hadoop.hdfs.server.namenode.FSPermissionChecker.checkPermissionWithContext(FSPermissionChecker.java:360)
	at org.apache.hadoop.hdfs.server.namenode.FSPermissionChecker.checkPermission(FSPermissionChecker.java:239)
	at org.apache.hadoop.hdfs.server.namenode.FSDirectory.checkPermission(FSDirectory.java:1909)
	at org.apache.hadoop.hdfs.server.namenode.FSDirDeleteOp.delete(FSDirDeleteOp.java:105)
	at org.apache.hadoop.hdfs.server.namenode.FSNamesystem.delete(FSNamesystem.java:3245)
	at org.apache.hadoop.hdfs.server.namenode.NameNodeRpcServer.delete(NameNodeRpcSer


### AccessControlException: Permission denied: user={username}, access=WRITE, inode="/user/hive/warehouse":hive:user:drwxr-xr-x
### Hive warehouse /user/{username}/warehouse is set as /user/{username}/warehouse
####  I had previously set the metastore at hive.metastore.warehouse.dir=/user/{username}/warehouse;

### Let's choose a database first and see if the location spark is trying to write to resolved

In [12]:
spark.sql("use big_data_025320")

""


In [13]:
spark.sql("show tables")

database,tableName,isTemporary
big_data_025320,orders_external,false
,order_view,true


In [14]:
spark.sql("""
    CREATE TABLE external_table_order_relative
    USING CSV
    LOCATION 'data'
    AS SELECT * from order_view
""")

""


In [16]:
spark.sql("describe extended external_table_order_relative").show(truncate=False)

+----------------------------+------------------------------------------------------------------------------+-------+
|col_name                    |data_type                                                                     |comment|
+----------------------------+------------------------------------------------------------------------------+-------+
|order_id                    |int                                                                           |null   |
|order_date                  |timestamp                                                                     |null   |
|customer_id                 |bigint                                                                        |null   |
|order_status                |string                                                                        |null   |
|                            |                                                                              |       |
|# Detailed Table Information|                          

In [18]:
spark.sql("show tables").show(truncate=False)

+---------------+-----------------------------+-----------+
|database       |tableName                    |isTemporary|
+---------------+-----------------------------+-----------+
|big_data_025320|external_table_order_relative|false      |
|big_data_025320|orders_external              |false      |
|               |order_view                   |true       |
+---------------+-----------------------------+-----------+



### When I give relative path without 'use database' then spark tries to write at default 'user/hive/warehouse' location which I don't have permission. So when I 'use database' I get path to the warehouse where I had created DB and that means I had access to that path and therefore I could create table within that chosen DB. 